In [6]:
%load_ext autoreload

In [ ]:
%autoreload 2
import os
import logging
import torch
import numpy as np
import contextlib
import patchcore
import mvtec as mvtec_dataset
import metrics
import common
import time
import statistics

logging.basicConfig(level=logging.INFO)
LOGGER = logging.getLogger(__name__)

def evaluate_patchcore_from_checkpoint(
    checkpoint_path,
    data_path,
    subdataset,
    batch_size=2,
    resize=256,
    imagesize=224,
    num_workers=8,
    gpu_id=0,
    num_timing_runs=5 
):
  
    device = torch.device(f"cuda:{gpu_id}" if torch.cuda.is_available() else "cpu")
    LOGGER.info(f"Using device: {device}")
    

    test_dataset = mvtec_dataset.MVTecDataset(
        data_path,
        classname=subdataset,
        resize=resize,
        imagesize=imagesize,
        split=mvtec_dataset.DatasetSplit.TEST,
    )

    test_dataloader = torch.utils.data.DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
    )
    
    with torch.cuda.device(f"cuda:{gpu_id}") if torch.cuda.is_available() else contextlib.suppress():

        start_load_time = time.time()
        patchcore_instance = patchcore.PatchCore(device)

        nn_method = common.FaissNN(True, 8)
        

        model_path = os.path.join(checkpoint_path, subdataset)
        LOGGER.info(f"Loading model from {model_path}")
        patchcore_instance.load_from_path(model_path, device=device, nn_method=nn_method)
        load_time = time.time() - start_load_time
        LOGGER.info(f"Model loading time: {load_time:.4f} seconds")
        

        LOGGER.info(f"Evaluating model for {subdataset}")
        start_time = time.time()
        scores, segmentations, labels_gt, masks_gt = patchcore_instance.predict(test_dataloader)
        first_run_time = time.time() - start_time
        num_samples = len(test_dataset)
        
     
        LOGGER.info(f"Collecting timing statistics over {num_timing_runs} runs...")
        inference_times_ms = []
        
 
        inference_times_ms.append((first_run_time / num_samples) * 1000)
       
        for run in range(1, num_timing_runs):
            start_time = time.time()
            _, _, _, _ = patchcore_instance.predict(test_dataloader)
            run_time = time.time() - start_time
            per_sample_time_ms = (run_time / num_samples) * 1000
            inference_times_ms.append(per_sample_time_ms)
            LOGGER.info(f"Run {run+1}/{num_timing_runs}: {per_sample_time_ms:.2f} ms per image")
        
 
        min_time_ms = min(inference_times_ms)
        max_time_ms = max(inference_times_ms)
        avg_time_ms = sum(inference_times_ms) / len(inference_times_ms)
        std_dev_ms = statistics.stdev(inference_times_ms) if len(inference_times_ms) > 1 else 0
        fps = 1000.0 / avg_time_ms
        
  
        LOGGER.info(f"===== Timing Statistics for {subdataset} =====")
        LOGGER.info(f"Min inference time per image: {min_time_ms:.2f} ms")
        LOGGER.info(f"Max inference time per image: {max_time_ms:.2f} ms")
        LOGGER.info(f"Avg inference time per image: {avg_time_ms:.2f} ms")
        LOGGER.info(f"Std deviation: {std_dev_ms:.2f} ms")
        LOGGER.info(f"FPS: {fps:.2f}")
        LOGGER.info(f"Number of images: {num_samples}")
        

        anomaly_labels = [x[1] != "good" for x in test_dataloader.dataset.data_to_iterate]
        

        image_metrics = metrics.compute_imagewise_retrieval_metrics(scores, anomaly_labels)

        pixel_metrics = metrics.compute_pixelwise_retrieval_metrics(segmentations, masks_gt)

        pro_metrics = metrics.compute_pro_metric(segmentations, masks_gt)

        sel_idxs = [i for i in range(len(masks_gt)) if np.sum(masks_gt[i]) > 0]
        
        anomaly_metrics = {}
        if len(sel_idxs) > 0:
            anomaly_pixel_metrics = metrics.compute_pixelwise_retrieval_metrics(
                [segmentations[i] for i in sel_idxs],
                [masks_gt[i] for i in sel_idxs],
            )
            
            anomaly_pro_metrics = metrics.compute_pro_metric(
                [segmentations[i] for i in sel_idxs],
                [masks_gt[i] for i in sel_idxs],
            )
            
            anomaly_metrics = {
                "anomaly_pixel_auroc": anomaly_pixel_metrics["auroc"],
                "anomaly_pixel_ap": anomaly_pixel_metrics["ap"],
                "anomaly_pixel_aupro": anomaly_pro_metrics["aupro"],
            }
        else:
            anomaly_metrics = {
                "anomaly_pixel_auroc": 0,
                "anomaly_pixel_ap": 0,
                "anomaly_pixel_aupro": 0,
            }
        

        timing_metrics = {
            "min_inference_time_ms": min_time_ms,
            "max_inference_time_ms": max_time_ms,
            "avg_inference_time_ms": avg_time_ms,
            "std_dev_ms": std_dev_ms,
            "fps": fps,
            "total_time_seconds": first_run_time
        }
        

        results = {
            "dataset": subdataset,
            "image_auroc": image_metrics["auroc"],
            "image_ap": image_metrics["ap"],
            "pixel_auroc": pixel_metrics["auroc"],
            "pixel_ap": pixel_metrics["ap"],
            "pixel_aupro": pro_metrics["aupro"],
            **anomaly_metrics,
            **timing_metrics
        }
        

        LOGGER.info(f"===== Results for {subdataset} =====")
        for key, value in results.items():
            if key != "dataset":
                LOGGER.info(f"{key}: {value:.4f}")
        
        return results

if __name__ == "__main__":

    results_front = evaluate_patchcore_from_checkpoint(
        checkpoint_path=r"C:\Thesis\Models\patchcore\checkpoints\patchcore-project\patchcore-project\models",
        data_path=r"C:\Thesis\Data\paperclips_sorted",
        subdataset="paperclips_front",
        batch_size=2,
        gpu_id=0,
        num_timing_runs=5  
    )

    results_side = evaluate_patchcore_from_checkpoint(
        checkpoint_path=r"C:\Thesis\Models\patchcore\checkpoints\patchcore-project\patchcore-group_0\models",
        data_path=r"C:\Thesis\Data\paperclips_sorted",
        subdataset="paperclips_side",
        batch_size=2,
        gpu_id=0,
        num_timing_runs=5  
    )
 
    LOGGER.info("\n===== Combined Timing Summary =====")
    LOGGER.info(f"Front view - Min: {results_front['min_inference_time_ms']:.2f}ms, Max: {results_front['max_inference_time_ms']:.2f}ms, Avg: {results_front['avg_inference_time_ms']:.2f}ms, Std: {results_front['std_dev_ms']:.2f}ms, FPS: {results_front['fps']:.2f}")
    LOGGER.info(f"Side view  - Min: {results_side['min_inference_time_ms']:.2f}ms, Max: {results_side['max_inference_time_ms']:.2f}ms, Avg: {results_side['avg_inference_time_ms']:.2f}ms, Std: {results_side['std_dev_ms']:.2f}ms, FPS: {results_side['fps']:.2f}")
    LOGGER.info(f"Average FPS across views: {(results_front['fps'] + results_side['fps'])/2:.2f}")

INFO:__main__:Using device: cuda:0
c:\Users\feder\anaconda3\envs\patchcore-env\lib\site-packages\torch\utils\data\dataloader.py:557: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 6 (`cpuset` is not taken into account), which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(
INFO:__main__:Loading model from C:\Thesis\Models\patchcore\checkpoints\patchcore-project\patchcore-project\models\paperclips_front
INFO:patchcore:Loading and initializing PatchCore.
c:\Users\feder\anaconda3\envs\patchcore-env\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Use